## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '.')

from multi_persona_explainer import (
    load_model_artifacts, get_claim_context, generate_explanation, explain_claim
)
import pandas as pd
import numpy as np
from IPython.display import Markdown, display

artifacts = load_model_artifacts()
claims_df = artifacts['claims_df']
print(f'Loaded {len(claims_df)} claims')
print(f'Declined: {(claims_df["target"]==0).sum()}, Completed: {(claims_df["target"]==1).sum()}')

---
## 2. Demo — Declined Claim

Let's explain a claim that was **declined**. This is the most important case — customers want to know *why*.

In [ ]:
# Find a declined claim with interesting characteristics
declined_indices = claims_df[claims_df['target'] == 0].index.tolist()
declined_idx = declined_indices[0]

ctx = get_claim_context(artifacts, declined_idx)
print(f'Claim #{declined_idx}')
print(f'  Prediction: {ctx["prediction"]} (P(approved) = {ctx["approval_probability"]:.3f})')
print(f'  Actual: {ctx["actual_status"]}')
print(f'  Type: {ctx["claim_type"]} | Coverage: {ctx["coverage"]}')
print(f'  Device: {ctx["make"]} {ctx["model_name"]} ({ctx["device_type"]})')
print(f'  Damage: {ctx["llm_damage_type"]} ({ctx["llm_damage_severity"]})')
print(f'  Incident: {ctx["llm_incident_type"]} | Gradual wear: {ctx["llm_is_gradual_wear"]}')
print(f'\n  Description: {ctx["issue_description"][:300]}...')
print(f'\n  Top SHAP factors:')
for f in ctx['top_factors'][:5]:
    print(f'    {f["feature"]}: {f["shap_value"]:+.4f} ({f["direction"]})')

In [ ]:
# Customer explanation
customer_exp = generate_explanation(ctx, 'customer')
display(Markdown(f'### Customer Explanation\n\n{customer_exp}'))

In [ ]:
# Adjuster explanation
adjuster_exp = generate_explanation(ctx, 'adjuster')
display(Markdown(f'### Claims Adjuster Note\n\n{adjuster_exp}'))

In [ ]:
# Manager explanation
manager_exp = generate_explanation(ctx, 'manager')
display(Markdown(f'### Manager Summary\n\n{manager_exp}'))

---
## 3. Demo — Approved Claim

A straightforward approved claim for comparison.

In [ ]:
# Find an approved claim with high confidence
approved_indices = claims_df[claims_df['target'] == 1].index.tolist()
approved_idx = approved_indices[5]  # Pick one that's not the first

ctx_approved = get_claim_context(artifacts, approved_idx)
print(f'Claim #{approved_idx}')
print(f'  Prediction: {ctx_approved["prediction"]} (P(approved) = {ctx_approved["approval_probability"]:.3f})')
print(f'  Type: {ctx_approved["claim_type"]} | Damage: {ctx_approved["llm_damage_type"]}')
print(f'  Incident: {ctx_approved["llm_incident_type"]} | Gradual wear: {ctx_approved["llm_is_gradual_wear"]}')

In [ ]:
result_approved = explain_claim(artifacts, approved_idx)

for persona in ['customer', 'adjuster', 'manager']:
    display(Markdown(f'### {persona.title()} Explanation\n\n{result_approved[persona]}'))
    print()

---
## 4. Demo — Borderline Case

A claim near the decision threshold — the most interesting case for adjuster review.

In [ ]:
# Find a borderline claim (probability close to threshold)
config = artifacts['config']
threshold = config['optimal_threshold']
feature_cols = config['feature_columns']

X_all = claims_df[feature_cols].fillna(0).values
probas = artifacts['model'].predict_proba(X_all)[:, 1]

# Find claims closest to threshold
distances = np.abs(probas - threshold)
borderline_indices = np.argsort(distances)[:10]
borderline_idx = int(borderline_indices[0])

ctx_border = get_claim_context(artifacts, borderline_idx)
print(f'Claim #{borderline_idx} (borderline)')
print(f'  Prediction: {ctx_border["prediction"]} (P(approved) = {ctx_border["approval_probability"]:.3f}, threshold = {threshold})')
print(f'  Actual: {ctx_border["actual_status"]}')
print(f'  Type: {ctx_border["claim_type"]} | Damage: {ctx_border["llm_damage_type"]}')
print(f'  This claim is RIGHT at the decision boundary.')

In [ ]:
result_border = explain_claim(artifacts, borderline_idx)

for persona in ['customer', 'adjuster', 'manager']:
    display(Markdown(f'### {persona.title()} Explanation (Borderline)\n\n{result_border[persona]}'))
    print()

---
## 5. Prompt Engineering Strategy

### Why Three Personas?

Each stakeholder needs different information from the same prediction:

| Persona | Needs | Tone | Detail Level | Jargon |
|---------|-------|------|-------------|--------|
| **Customer** | Why was my claim approved/declined? What can I do? | Empathetic, clear | Low — plain language | None |
| **Claims Adjuster** | Is this prediction trustworthy? What evidence supports it? | Technical, analytical | High — specific data points | Insurance + ML terms OK |
| **Manager** | Should I review this? What's the risk? | Concise, executive | Summary only | Business terms |

### Grounding Strategy

Every explanation is grounded in **three layers of evidence**:

1. **SHAP values** — which features pushed the model toward approval or decline, and by how much
2. **LLM-extracted features** — structured damage assessment (type, severity, gradual wear, etc.)
3. **Raw claim data** — device info, policy details, financial data

This prevents hallucination — the LLM explains what the model actually decided, not what it imagines.

### Prompt Design Principles

1. **Role assignment**: Each prompt starts with a clear persona ("You are a friendly customer service representative" vs "You are writing an internal adjuster note")
2. **Context injection**: Claim data, SHAP factors, and LLM features are injected as structured data
3. **Output constraints**: Word limits, format requirements (bullets for adjuster, paragraphs for customer)
4. **Actionability**: Customer prompts require next-step guidance; adjuster prompts require risk flags
5. **Guardrails**: No SHAP numbers in customer output; no emotional language in adjuster output

---
## 6. Batch Generation — Sample Across Scenarios

In [ ]:
# Generate explanations for a diverse set of claims
np.random.seed(42)

# Pick: 3 declined, 3 approved, 2 borderline
declined_sample = np.random.choice(declined_indices, 3, replace=False)
approved_sample = np.random.choice(approved_indices, 3, replace=False)
borderline_sample = borderline_indices[:2]

sample_indices = list(declined_sample) + list(approved_sample) + list(borderline_sample)

batch_results = []
for idx in sample_indices:
    idx = int(idx)
    result = explain_claim(artifacts, idx, personas=['customer', 'adjuster'])
    ctx = result['claim_context']
    batch_results.append({
        'claim_idx': idx,
        'prediction': ctx['prediction'],
        'actual': ctx['actual_status'],
        'probability': ctx['approval_probability'],
        'claim_type': ctx['claim_type'],
        'damage_type': ctx['llm_damage_type'],
        'customer_explanation': result['customer'],
        'adjuster_explanation': result['adjuster'],
    })
    print(f'  Claim #{idx}: {ctx["prediction"]} (P={ctx["approval_probability"]:.3f})')

batch_df = pd.DataFrame(batch_results)
print(f'\nGenerated {len(batch_df)} explanations')

In [ ]:
# Display a few examples
for _, row in batch_df.head(4).iterrows():
    display(Markdown(f'---\n### Claim #{row["claim_idx"]} — {row["prediction"]} ({row["claim_type"]}, {row["damage_type"]})'))
    display(Markdown(f'**Customer:**\n\n{row["customer_explanation"]}'))
    display(Markdown(f'**Adjuster:**\n\n{row["adjuster_explanation"]}'))

---
## 7. Save Outputs

In [ ]:
from pathlib import Path
output_dir = Path('../../data')
batch_df.to_csv(output_dir / 'sample_explanations.csv', index=False)
print(f'Saved sample explanations to {output_dir / "sample_explanations.csv"}')
print(f'Columns: {list(batch_df.columns)}')

---
## Wrap up

The explanation pipeline works. For each claim it pulls the SHAP values and LLM-extracted features, builds a persona-specific prompt, and sends it to Claude. Three personas:
- Customer gets a friendly explanation with next steps
- Adjuster gets SHAP numbers, evidence bullets, and risk flags
- Manager gets a 3-sentence summary with escalation recommendation

The key thing is grounding - every explanation references actual model outputs, not made-up reasoning. If SHAP says incident_type pushed toward decline, the explanation mentions that.

The `multi_persona_explainer.py` module is designed to plug straight into the FastAPI endpoint.